# Oracle Database Security Learning Lab
### Hands-on Database Defense-in-Depth

This notebook serves as an interactive report and training manual for the Oracle Database Security Laboratory. We will execute 9 security modules to identify vulnerabilities and implement enterprise-grade defenses.

---

## 0. Laboratory Setup
First, we initialize the database with realistic data and "security smells" (vulnerabilities).

In [2]:
!docker exec -i oracle-security-lab-app python lab/core/setup_db.py

Starting Database Initialization...
Connected to Oracle SYSDBA.
Setting up HR schema (2,000 records)...
Error setting up HR: ORA-01920: user name 'HR' conflicts with another user or 
role name
Help: https://docs.oracle.com/error-help/db/ora-01920/
Setting up CORPORATE_PII schema (5,000 records)...
Error setting up CORPORATE_PII: ORA-01920: user name 'CORPORATE_PII' conflicts 
with another user or role name
Help: https://docs.oracle.com/error-help/db/ora-01920/
Injecting realistic security smells...
Error injecting smells: ORA-01920: user name 'WEAK_USER' conflicts with another 
user or role name
Help: https://docs.oracle.com/error-help/db/ora-01920/
Initialization Complete!


## 1. Reconnaissance
Understanding the target environment and identifying active security features.

In [3]:
!docker exec -i oracle-security-lab-app python lab/01_recon.py

Module 01: Reconnaissance

Database Banner: Oracle Database 21c Express Edition Release 21.0.0.0.0 - 
Production
          Instance Status           
┏━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Instance ┃ Host         ┃ Status ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━┩
│ XE       │ 99c9361521f9 │ OPEN   │
└──────────┴──────────────┴────────┘
              Security Options               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Feature               ┃ Installed/Enabled ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Oracle Label Security │ FALSE             │
│ Oracle Database Vault │ FALSE             │
│ Unified Auditing      │ FALSE             │
└───────────────────────┴───────────────────┘


## 2. RBAC & Privilege Auditing
Scanning for over-privileged accounts, Shadow DBAs, and data exposed to `PUBLIC`.

In [4]:
!docker exec -i oracle-security-lab-app python lab/02_rbac_audit.py

Module 02: RBAC Audit (Identity & Access Management)
  Users with DBA Role   
┏━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Grantee  ┃ Admin Opt ┃
┡━━━━━━━━━━╇━━━━━━━━━━━┩
│ IT_ADMIN │ NO        │
│ SYS      │ YES       │
│ SYSTEM   │ NO        │
└──────────┴───────────┘
                  Dangerous 'ANY' Privileges                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Grantee                    ┃ Privilege                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ APP_SUPPORT                │ SELECT ANY TABLE              │
│ APP_SUPPORT                │ DROP ANY TABLE                │
│ APP_SUPPORT                │ CREATE ANY TABLE              │
│ AQ_ADMINISTRATOR_ROLE      │ DEQUEUE ANY QUEUE             │
│ AQ_ADMINISTRATOR_ROLE      │ MANAGE ANY QUEUE              │
│ AQ_ADMINISTRATOR_ROLE      │ ENQUEUE ANY QUEUE             │
│ AUDSYS                     │ SELECT ANY DICTIONARY         │
│ CTXSYS                     │ INHERIT ANY PRIVILEGES 

## 3. Unified Auditing
Implementing a unified audit policy to track access to sensitive PII data.

In [5]:
!docker exec -i oracle-security-lab-app python lab/03_audit_trail.py

Module 03: Unified Auditing
Creating Audit Policy: AUDIT_PII_ACCESS...
Error setting up audit policy: ORA-46358: Audit policy AUDIT_PII_ACCESS already 
exists.
Help: https://docs.oracle.com/error-help/db/ora-46358/
Simulating access to CORPORATE_PII.CLIENT_RECORDS...
Waiting for audit trail flush...
                         Recent Audit Logs (PII Access)                         
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Timestamp          ┃ User          ┃ Action ┃ Schema        ┃ Object         ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ 2026-04-23         │ CORPORATE_PII │ SELECT │ CORPORATE_PII │ CLIENT_RECORDS │
│ 20:56:04.953427    │               │        │               │                │
│ 2026-04-23         │ CORPORATE_PII │ SELECT │ CORPORATE_PII │ CLIENT_RECORDS │
│ 14:31:13.962571    │               │        │               │                │
│ 2026-04-23         │ CORPORATE_PII │ SELECT │ COR

## 4. Data Redaction
On-the-fly masking of sensitive columns (SSN and Credit Cards) for application users.

In [6]:
!docker exec -i oracle-security-lab-app python lab/04_data_masking.py

Module 04: Data Redaction
Redaction policy already exists.

Verifying Data Redaction as CORPORATE_PII user...
                      Redacted Data View                      
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name              ┃ SSN (Redacted) ┃ Credit Card (Partial) ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ James Perez       │                │ ************14        │
│ Alexandra Vazquez │                │ ************06        │
│ Brian Ayers       │                │ ************200       │
│ Steven Garcia     │                │ ************2329      │
│ David Harris      │                │ ************10        │
└───────────────────┴────────────────┴───────────────────────┘


## 5. SQL Injection & Secure Coding
Demonstrating a classic exploit and its mitigation using bind variables.

In [7]:
!docker exec -i oracle-security-lab-app python lab/05_sql_injection.py

Module 05: SQL Injection & Secure Coding

1. Standard Search (Bonnie):
Executing Secure SQL: SELECT first_name, last_name, salary FROM HR.EMPLOYEES 
WHERE first_name = :name with bind :name='Bonnie'
[('Bonnie', 'Thompson', 45978.0)]

2. Attempting SQL Injection on Vulnerable Function:
Attack Payload: Alice' OR '1'='1
Executing Vulnerable SQL: SELECT first_name, last_name, salary FROM HR.EMPLOYEES
WHERE first_name = 'Alice' OR '1'='1'
Result count (Expected 1, got all): 2000

3. Attempting SQL Injection on Secure Function:
Executing Secure SQL: SELECT first_name, last_name, salary FROM HR.EMPLOYEES 
WHERE first_name = :name with bind :name='Alice' OR '1'='1'
Result count (Correctly got 0): 0


## 6. Row-Level Security (VPD)
Using Virtual Private Database to restrict users to seeing only their own department's data.

In [8]:
!docker exec -i oracle-security-lab-app python lab/06_row_level_security.py

Module 06: Virtual Private Database (VPD)
Setting up VPD Policy...
VPD Policy 'DEPT_ACCESS_POLICY' is now active on HR.EMPLOYEES.

Simulating Session: Sales Manager (Authorized for Dept 10)
    Visible Records for Dept 10     
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┓
┃ First Name ┃ Last Name ┃ Dept ID ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━┩
│ Thomas     │ Morrison  │ 10      │
│ Douglas    │ Lowery    │ 10      │
│ Kyle       │ Griffith  │ 10      │
└────────────┴───────────┴─────────┘
Total Rows Visible: 500 (Out of 2,000 total)

Simulating Session: Engineering Lead (Authorized for Dept 30)
     Visible Records for Dept 30     
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┓
┃ First Name  ┃ Last Name ┃ Dept ID ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━┩
│ David       │ Phillips  │ 30      │
│ Leslie      │ Nelson    │ 30      │
│ Christopher │ Ewing     │ 30      │
└─────────────┴───────────┴─────────┘
Total Rows Visible: 500 (Out of 2,000 total)

Simulating Session: IT_ADMIN (Bypassing VPD)
Total Rows 

## 7. Transparent Data Encryption (TDE)
Encrypting data at rest at the storage level to protect against physical data theft.

In [9]:
!docker exec -i oracle-security-lab-app python lab/07_tde.py

Module 07: Transparent Data Encryption (TDE)
Setting up Transparent Data Encryption (TDE)...
Connecting to CDB Root for Keystore Management...
Creating/Opening keystore in CDB...
Keystore already exists or location is initialized.
Keystore already open.
Opening keystore in PDB (XEPDB1)...
Keystore already open in PDB.
Setting master encryption key in PDB...
Encrypting CORPORATE_PII.CLIENT_RECORDS(SSN) column...
Column already encrypted.
TDE is now active. The SSN column is encrypted at rest.
Verified: SSN is encrypted with AES 192 bits key


## 8. Fine-Grained Auditing (The Admin Trap)
Implementing stealth monitoring to catch administrative users accessing sensitive application data.

In [10]:
!docker exec -i oracle-security-lab-app python lab/08_fga_trap.py

Module 08: Fine-Grained Auditing (FGA) - The 'Admin Trap'
Setting up Fine-Grained Auditing (FGA) 'Admin Trap'...
FGA Policy 'CATCH_DBA_PEEKING' is now active.
This policy monitors any non-owner access to sensitive PII columns.

Scenario: IT_ADMIN (DBA) is peeking at PII data...
Admin successfully retrieved data (Redaction might still be active, but FGA 
caught the attempt).
Waiting for audit trail flush (5s)...

Checking FGA Audit Logs (The Trap Result):
                          FGA Intrusion Detection Logs                          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Timestamp           ┃ Intruder ┃ SQL Statement                               ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2026-04-23 20:56:12 │ IT_ADMIN │ SELECT full_name, ssn FROM                  │
│                     │          │ CORPORATE_PII.CLIENT_RE...                  │
│ 2026-04-23 14:31:18 │ IT_ADMIN │ SELECT full_name, ss

## 9. Secure Backup & Resiliency (ZDLRA Simulation)
Verifying that the database is in ARCHIVELOG mode and that backups are encrypted using the TDE wallet.

In [11]:
!docker exec -i oracle-security-lab-app python lab/09_secure_backup.py

Module 09: Secure Backup & Resiliency (ZDLRA Simulation)
Verifying RMAN Secure Backup Configuration...
Database Log Mode: ARCHIVELOG
RMAN Config: ENCRYPTION FOR DATABASE = ON
RMAN Config: ENCRYPTION ALGORITHM = 'AES256'

Auditing Backup Encryption Status (ZDLRA Proof):
                         Audit: Encrypted Backup Pieces                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Backup Piece (Handle)                     ┃ Encrypted? ┃ Completion Time     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ /opt/oracle/homes/OraDBHome21cXE/dbs/c-3… │ YES        │ 2026-04-23 20:49:40 │
│ /opt/oracle/homes/OraDBHome21cXE/dbs/024… │ YES        │ 2026-04-23 20:49:39 │
└───────────────────────────────────────────┴────────────┴─────────────────────┘

VALIDATION SUCCESSFUL:
1. Real-time logging (ARCHIVELOG) is active.
2. Backups are encrypted at rest using AES-256.
3. Recovery vault integrity is verified.


---
**End of Security Audit Report**